In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/CBCT_to_sCT_Project

Mounted at /content/drive
/content/drive/MyDrive/CBCT_to_sCT_Project


In [2]:
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
os.makedirs("results/final_project_figures", exist_ok=True)
os.makedirs("results/final_tables", exist_ok=True)

In [4]:
final_summary = pd.read_csv(
    "results/metrics/final_single_fold_summary.csv",
    index_col=0
)

final_improvement = pd.read_csv(
    "results/metrics/final_single_fold_improvement_percent.csv"
)

masked_ssim = pd.read_csv(
    "results/metrics/final_masked_background_ssim.csv"
)

mc_random_summary = pd.read_csv(
    "results/metrics/mc_dropout_uncertainty_summary_random100.csv",
    index_col=0
)

final_summary

,mean,std
raw_cbct_mae_hu,216.972275,84.029968
sct_mae_hu,88.842125,33.060631
raw_cbct_rmse_hu,271.228790,78.407684
sct_rmse_hu,170.498596,54.418419
raw_cbct_psnr,19.988859,2.509677
sct_psnr,24.081551,2.682746
raw_cbct_ssim,0.653904,0.118190
sct_ssim,0.330198,0.040281
raw_cbct_mae_body_hu,216.972275,84.029968
sct_mae_body_hu,88.842125,33.060631


In [5]:
final_table = pd.DataFrame({
    "Metric": [
        "MAE [HU]",
        "RMSE [HU]",
        "PSNR",
        "SSIM",
        "Body MAE [HU]",
        "Bone MAE [HU]",
        "Soft tissue MAE [HU]",
        "Air/lung MAE [HU]"
    ],
    "Raw CBCT": [
        final_summary.loc["raw_cbct_mae_hu", "mean"],
        final_summary.loc["raw_cbct_rmse_hu", "mean"],
        final_summary.loc["raw_cbct_psnr", "mean"],
        final_summary.loc["raw_cbct_ssim", "mean"],
        final_summary.loc["raw_cbct_mae_body_hu", "mean"],
        final_summary.loc["raw_cbct_mae_bone_hu", "mean"],
        final_summary.loc["raw_cbct_mae_soft_tissue_hu", "mean"],
        final_summary.loc["raw_cbct_mae_air_lung_hu", "mean"],
    ],
    "Final sCT": [
        final_summary.loc["sct_mae_hu", "mean"],
        final_summary.loc["sct_rmse_hu", "mean"],
        final_summary.loc["sct_psnr", "mean"],
        final_summary.loc["sct_ssim", "mean"],
        final_summary.loc["sct_mae_body_hu", "mean"],
        final_summary.loc["sct_mae_bone_hu", "mean"],
        final_summary.loc["sct_mae_soft_tissue_hu", "mean"],
        final_summary.loc["sct_mae_air_lung_hu", "mean"],
    ]
})

final_table["Improvement"] = final_table["Raw CBCT"] - final_table["Final sCT"]

final_table.to_csv("results/final_tables/final_results_summary.csv", index=False)

final_table

,Metric,Raw CBCT,Final sCT,Improvement
0,MAE [HU],216.972275,88.842125,128.130150
1,RMSE [HU],271.228790,170.498596,100.730194
2,PSNR,19.988859,24.081551,-4.092691
3,SSIM,0.653904,0.330198,0.323706
4,Body MAE [HU],216.972275,88.842125,128.130150
5,Bone MAE [HU],391.343323,213.658340,177.684982
6,Soft tissue MAE [HU],210.855118,60.591961,150.263157
7,Air/lung MAE [HU],153.813858,186.411591,-32.597733


In [6]:
improvement_clean = final_improvement.T.reset_index()
improvement_clean.columns = ["Region", "MAE improvement [%]"]

improvement_clean["Region"] = improvement_clean["Region"].replace({
    "overall_mae_improvement_%": "Overall/body",
    "body_mae_improvement_%": "Body",
    "bone_mae_improvement_%": "Bone",
    "soft_tissue_mae_improvement_%": "Soft tissue",
    "air_lung_mae_improvement_%": "Air/lung"
})

improvement_clean.to_csv(
    "results/final_tables/final_improvement_percent.csv",
    index=False
)

improvement_clean

,Region,MAE improvement [%]
0,Overall/body,59.053700
1,Body,59.053700
2,Bone,45.403862
3,Soft tissue,71.263695
4,Air/lung,-21.192974


In [7]:
day10_df = pd.read_csv("results/metrics/day11_regionwise_metrics.csv")
day12_df = pd.read_csv("results/metrics/day12_first300_validation_metrics.csv")
day13_df = pd.read_csv("results/metrics/day13_partial_first300_validation_metrics.csv")
final_df = pd.read_csv("results/metrics/final_single_fold_validation_metrics.csv")
mc_df = pd.read_csv("results/metrics/mc_dropout_uncertainty_metrics_random100.csv")

ablation_table = pd.DataFrame({
    "Experiment": [
        "Day 10: Combined loss, 30 train patients",
        "Day 12: Region-weighted loss",
        "Day 13: Combined loss, 80 train patients partial",
        "Final deterministic model",
        "MC dropout mean prediction, random 100"
    ],
    "sCT MAE [HU]": [
        day10_df["sct_mae_hu"].mean(),
        day12_df["sct_mae_hu"].mean(),
        day13_df["sct_mae_hu"].mean(),
        final_df["sct_mae_hu"].mean(),
        mc_df["sct_mae_hu"].mean()
    ],
    "sCT RMSE [HU]": [
        day10_df["sct_rmse_hu"].mean(),
        day12_df["sct_rmse_hu"].mean(),
        day13_df["sct_rmse_hu"].mean(),
        final_df["sct_rmse_hu"].mean(),
        mc_df["sct_rmse_hu"].mean()
    ],
    "sCT PSNR": [
        day10_df["sct_psnr"].mean(),
        day12_df["sct_psnr"].mean(),
        day13_df["sct_psnr"].mean(),
        final_df["sct_psnr"].mean(),
        mc_df["sct_psnr"].mean()
    ],
    "sCT SSIM": [
        day10_df["sct_ssim"].mean(),
        day12_df["sct_ssim"].mean(),
        day13_df["sct_ssim"].mean(),
        final_df["sct_ssim"].mean(),
        mc_df["sct_ssim"].mean()
    ]
})

ablation_table.to_csv(
    "results/final_tables/final_ablation_summary.csv",
    index=False
)

ablation_table

,Experiment,sCT MAE [HU],sCT RMSE [HU],sCT PSNR,sCT SSIM
0,"Day 10: Combined loss, 30 train patients",91.435418,177.767331,23.578048,0.394469
1,Day 12: Region-weighted loss,96.751906,181.885038,23.374651,0.358966
2,"Day 13: Combined loss, 80 train patients partial",92.691748,180.689010,23.471625,0.330230
3,Final deterministic model,88.842131,170.498582,24.081551,0.330198
4,"MC dropout mean prediction, random 100",97.216781,183.418941,23.393901,0.314663


In [8]:
figures_to_copy = {
    "final_training_curve.png": "results/figures/final_model/final_training_curve.png",
    "final_raw_vs_sct_mae.png": "results/figures/final_model/final_raw_vs_sct_mae.png",
    "final_regionwise_mae.png": "results/figures/final_model/final_regionwise_mae.png",
    "final_example_1.png": "results/figures/final_model/final_example_1.png",
    "final_example_2.png": "results/figures/final_model/final_example_2.png",
    "final_example_3.png": "results/figures/final_model/final_example_3.png",
    "mc_dropout_example.png": "results/figures/uncertainty/mc_dropout_example.png",
    "random100_uncertainty_vs_error_scatter.png": "results/figures/uncertainty/random100_uncertainty_vs_error_scatter.png",
    "random100_most_uncertain_case.png": "results/figures/uncertainty/random100_most_uncertain_case.png"
}

for new_name, old_path in figures_to_copy.items():
    if os.path.exists(old_path):
        shutil.copy(old_path, f"results/final_project_figures/{new_name}")
        print("Copied:", new_name)
    else:
        print("Missing:", old_path)

Copied: final_training_curve.png
Copied: final_raw_vs_sct_mae.png
Copied: final_regionwise_mae.png
Copied: final_example_1.png
Copied: final_example_2.png
Copied: final_example_3.png
Copied: mc_dropout_example.png
Copied: random100_uncertainty_vs_error_scatter.png
Copied: random100_most_uncertain_case.png


In [9]:
!ls results/final_project_figures

final_example_1.png	  final_training_curve.png
final_example_2.png	  mc_dropout_example.png
final_example_3.png	  random100_most_uncertain_case.png
final_raw_vs_sct_mae.png  random100_uncertainty_vs_error_scatter.png
final_regionwise_mae.png


In [10]:
summary_text = """
# CBCT-to-synthetic CT Generation Using 2.5D U-Net++

## Project Goal
This project develops a CBCT-to-synthetic CT pipeline for adaptive radiotherapy using SynthRAD2025 Task 2 data. The goal is to convert artifact-prone CBCT images into CT-like images with improved HU accuracy.

## Final Model
- Architecture: 2.5D U-Net++ with ResNet-34 encoder
- Input: 7 neighboring CBCT slices
- Output: center synthetic CT slice
- Image size: 320 x 320
- Loss: L1 + 0.2 SSIM + 0.05 Gradient loss
- Training: single fold, 80 training patients, 20 validation patients
- Epochs: 25
- Best epoch: 12

## Final Result
The final deterministic model reduced overall MAE from raw CBCT to synthetic CT by approximately 59%.

Key results:
- Raw CBCT MAE: 216.97 HU
- Final sCT MAE: 88.84 HU
- Overall MAE improvement: 59.05%
- Soft tissue MAE improvement: 71.26%
- Masked-background SSIM: approximately 0.798

## Uncertainty Analysis
Monte Carlo dropout was implemented using 20 stochastic forward passes per slice. Random-100 validation analysis showed a weak positive error-uncertainty correlation, suggesting exploratory but not fully calibrated uncertainty estimation.

## Main Limitation
The model improved body, bone, and soft tissue regions, but low-density air/lung regions remained challenging.

## Final Use
The deterministic model is used for final sCT generation. MC dropout is reported as exploratory uncertainty analysis.
"""

with open("results/final_tables/final_project_summary.txt", "w") as f:
    f.write(summary_text)

print(summary_text)


# CBCT-to-synthetic CT Generation Using 2.5D U-Net++

## Project Goal
This project develops a CBCT-to-synthetic CT pipeline for adaptive radiotherapy using SynthRAD2025 Task 2 data. The goal is to convert artifact-prone CBCT images into CT-like images with improved HU accuracy.

## Final Model
- Architecture: 2.5D U-Net++ with ResNet-34 encoder
- Input: 7 neighboring CBCT slices
- Output: center synthetic CT slice
- Image size: 320 x 320
- Loss: L1 + 0.2 SSIM + 0.05 Gradient loss
- Training: single fold, 80 training patients, 20 validation patients
- Epochs: 25
- Best epoch: 12

## Final Result
The final deterministic model reduced overall MAE from raw CBCT to synthetic CT by approximately 59%.

Key results:
- Raw CBCT MAE: 216.97 HU
- Final sCT MAE: 88.84 HU
- Overall MAE improvement: 59.05%
- Soft tissue MAE improvement: 71.26%
- Masked-background SSIM: approximately 0.798

## Uncertainty Analysis
Monte Carlo dropout was implemented using 20 stochastic forward passes per slice. Rand

In [11]:
%%writefile .gitignore
# Dataset
data/
*.mha
*.nii
*.nii.gz
*.dcm

# Model weights and checkpoints
checkpoints/
*.pth
*.pt
*.ckpt

# Large outputs
results/predictions/
results/volumes/

# Python cache
__pycache__/
.ipynb_checkpoints/
*.pyc

# System files
.DS_Store

Overwriting .gitignore
